# CSIRO Biomass — Tuned LightGBM + XGBoost Ensemble

**Approach:** Metadata-driven ensemble with rich feature engineering and Optuna-tuned per-target models.

### Key findings from EDA
- NDVI × Height interaction is the strongest predictor (SHAP #1)
- Metadata dominates over image embeddings in this dataset (CV 0.716 vs 0.677)
- Per-target tuning helps because targets have different drivers (clover, dead biomass differ from GDM)
- Blending LightGBM + XGBoost OOFs gives consistent ~0.005 improvement

### Target weights (competition metric)
| Target | Weight |
|--------|--------|
| Dry_Total_g | 0.5 |
| GDM_g | 0.2 |
| Dry_Green_g | 0.1 |
| Dry_Dead_g | 0.1 |
| Dry_Clover_g | 0.1 |

In [ ]:
import json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import optuna
from lightgbm import LGBMRegressor
from scipy.optimize import minimize
from sklearn.model_selection import KFold
from xgboost import XGBRegressor

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

DATA_DIR = Path('/kaggle/input/csiro-biomass-data')
OUT_DIR = Path('/kaggle/working')
SEED = 42
N_FOLDS = 5
N_TRIALS = 80

TARGET_COLS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
TARGET_WEIGHTS = {'Dry_Green_g': 0.1, 'Dry_Dead_g': 0.1, 'Dry_Clover_g': 0.1, 'GDM_g': 0.2, 'Dry_Total_g': 0.5}

print('Setup complete')

In [ ]:
# ── Feature Engineering ──────────────────────────────────────────────────────

_CLOVER_SPECIES = {'Clover', 'Ryegrass_Clover', 'WhiteClover', 'Phalaris_Clover',
                   'SubcloverLosa', 'SubcloverDalkeith'}

def build_features(df: pd.DataFrame, fit_stats: dict | None = None) -> tuple[pd.DataFrame, dict]:
    out = df.copy()
    stats = {} if fit_stats is None else dict(fit_stats)

    if 'Sampling_Date' in out.columns:
        dates = pd.to_datetime(out['Sampling_Date'], errors='coerce')
        out['month'] = dates.dt.month
        out['dayofyear'] = dates.dt.dayofyear
        out['year'] = dates.dt.year
        out['month_sin'] = np.sin(2 * np.pi * out['month'] / 12)
        out['month_cos'] = np.cos(2 * np.pi * out['month'] / 12)
        out['doy_sin'] = np.sin(2 * np.pi * out['dayofyear'] / 365)
        out['doy_cos'] = np.cos(2 * np.pi * out['dayofyear'] / 365)

    if 'Species' in out.columns:
        sp = out['Species'].fillna('unknown').astype(str)
        out['species_count'] = sp.str.split(r'[_,;/|+]').map(len)
        if fit_stats is None:
            stats['species_freq'] = sp.value_counts(normalize=True).to_dict()
        out['species_freq'] = sp.map(stats.get('species_freq', {})).fillna(0)
        out['has_clover'] = sp.isin(_CLOVER_SPECIES).astype(int)
        out['has_ryegrass'] = sp.str.contains('Ryegrass', na=False).astype(int)
        out['has_phalaris'] = sp.str.contains('Phalaris', na=False).astype(int)
        out['has_lucerne'] = sp.str.contains('Lucerne', na=False).astype(int)
        out['has_fescue'] = sp.str.contains('Fescue', na=False).astype(int)

    if 'Pre_GSHH_NDVI' in out.columns:
        n = out['Pre_GSHH_NDVI'].clip(lower=0)
        out['ndvi_sq'] = n ** 2
        out['ndvi_sqrt'] = np.sqrt(n)
        out['ndvi_log1p'] = np.log1p(n)

    if 'Height_Ave_cm' in out.columns:
        h = out['Height_Ave_cm'].clip(lower=0)
        out['height_sq'] = h ** 2
        out['height_sqrt'] = np.sqrt(h)
        out['height_log1p'] = np.log1p(h)

    if {'Pre_GSHH_NDVI', 'Height_Ave_cm'}.issubset(out.columns):
        n = out['Pre_GSHH_NDVI'].clip(lower=0)
        h = out['Height_Ave_cm'].clip(lower=0)
        out['ndvi_x_height'] = n * h
        out['height_per_ndvi'] = h / n.replace(0, np.nan)
        out['ndvi_x_height_sq'] = n * h ** 2
        out['ndvi_sq_x_height'] = n ** 2 * h

    if {'Pre_GSHH_NDVI', 'doy_sin'}.issubset(out.columns):
        out['ndvi_x_doy_sin'] = out['Pre_GSHH_NDVI'] * out['doy_sin']
        out['ndvi_x_doy_cos'] = out['Pre_GSHH_NDVI'] * out['doy_cos']

    if {'Height_Ave_cm', 'doy_sin'}.issubset(out.columns):
        out['height_x_doy_sin'] = out['Height_Ave_cm'] * out['doy_sin']

    if {'Pre_GSHH_NDVI', 'Height_Ave_cm', 'dayofyear'}.issubset(out.columns):
        out['ndvi_x_height_x_doy'] = (
            out['Pre_GSHH_NDVI'] * out['Height_Ave_cm'] * out['dayofyear'] / 365
        )

    return out, stats

print('Feature functions ready')

In [ ]:
# ── Load & pivot data ─────────────────────────────────────────────────────────

train_long = pd.read_csv(DATA_DIR / 'train.csv')
meta_cols = [c for c in train_long.columns if c not in {'target_name', 'target', 'sample_id', 'image_path'}]
meta = train_long.groupby('image_path')[meta_cols].first().reset_index()
targets_wide = train_long.pivot_table(
    index='image_path', columns='target_name', values='target', aggfunc='first'
).reset_index()
train = meta.merge(targets_wide, on='image_path')
train = train.dropna(subset=TARGET_COLS).reset_index(drop=True)
print(f'Train: {len(train)} images, {len(train.columns)} columns')
train[TARGET_COLS].describe().round(1)

In [ ]:
# ── Prepare features ──────────────────────────────────────────────────────────

feat_df, stats = build_features(train)
drop_cols = set(TARGET_COLS) | {'image_path', 'Sampling_Date', 'sample_id', 'target', 'target_name'}
x_df = feat_df[[c for c in feat_df.columns if c not in drop_cols]].copy()
y_df = feat_df[TARGET_COLS].copy()

# One-hot encode categoricals, impute numerics
cat_cols = [c for c in x_df.columns if x_df[c].dtype == object]
x_df = pd.get_dummies(x_df, columns=cat_cols, drop_first=False)
x_df = x_df.fillna(x_df.median(numeric_only=True))
X = x_df.values.astype(float)

folds = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_ids = np.zeros(len(train), dtype=int)
for i, (_, val) in enumerate(folds.split(X)):
    fold_ids[val] = i

print(f'Feature matrix: {X.shape}')
print(f'Features: {list(x_df.columns[:10])} ...')

In [ ]:
# ── Metric ────────────────────────────────────────────────────────────────────

def weighted_log_r2(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.log1p(np.clip(np.asarray(y_true, float), 0, None))
    y_pred = np.log1p(np.clip(np.asarray(y_pred, float), 0, None))
    score = 0.0
    for i, t in enumerate(TARGET_COLS):
        yt, yp = y_true[:, i], y_pred[:, i]
        denom = np.sum((yt - yt.mean()) ** 2)
        r2 = 0.0 if denom == 0 else 1.0 - np.sum((yt - yp) ** 2) / denom
        score += TARGET_WEIGHTS[t] * r2
    return float(score)

print('Metric ready')

In [ ]:
# ── Optuna: tune LightGBM per target ─────────────────────────────────────────

best_params = {}
for target in TARGET_COLS:
    y_col = y_df[target].values

    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 400, 3000),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 15, 127),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 60),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 2.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 5.0, log=True),
            'random_state': SEED, 'verbose': -1,
        }
        oof = np.zeros(len(y_col))
        for fold in range(N_FOLDS):
            trn, val = fold_ids != fold, fold_ids == fold
            m = LGBMRegressor(**params)
            m.fit(X[trn], np.log1p(y_col[trn].clip(0)))
            oof[val] = np.expm1(m.predict(X[val])).clip(0)
        yt = np.log1p(y_col.clip(0))
        yp = np.log1p(oof.clip(0))
        denom = np.sum((yt - yt.mean()) ** 2)
        return -(1.0 - np.sum((yt - yp) ** 2) / (denom + 1e-9))

    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
    best_params[target] = study.best_params
    print(f'  {target}: R2={-study.best_value:.4f}')

with open(OUT_DIR / 'best_lgbm_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)
print('\nTuning complete. Params saved.')

In [ ]:
# ── OOF: tuned LightGBM ───────────────────────────────────────────────────────

lgbm_oof = np.zeros((len(train), len(TARGET_COLS)))
for i, target in enumerate(TARGET_COLS):
    p = {**best_params[target], 'random_state': SEED, 'verbose': -1}
    y_col = y_df[target].values
    for fold in range(N_FOLDS):
        trn, val = fold_ids != fold, fold_ids == fold
        m = LGBMRegressor(**p)
        m.fit(X[trn], np.log1p(y_col[trn].clip(0)))
        lgbm_oof[val, i] = np.expm1(m.predict(X[val])).clip(0)

lgbm_cv = weighted_log_r2(y_df[TARGET_COLS].values, lgbm_oof)
print(f'Tuned LightGBM CV: {lgbm_cv:.6f}')

In [ ]:
# ── OOF: XGBoost ─────────────────────────────────────────────────────────────

xgb_oof = np.zeros((len(train), len(TARGET_COLS)))
for i, target in enumerate(TARGET_COLS):
    y_col = y_df[target].values
    for fold in range(N_FOLDS):
        trn, val = fold_ids != fold, fold_ids == fold
        m = XGBRegressor(
            n_estimators=1500, learning_rate=0.02, max_depth=5,
            min_child_weight=3, subsample=0.85, colsample_bytree=0.85,
            reg_alpha=0.1, reg_lambda=1.0, objective='reg:squarederror',
            random_state=SEED, verbosity=0,
        )
        m.fit(X[trn], np.log1p(y_col[trn].clip(0)))
        xgb_oof[val, i] = np.expm1(m.predict(X[val])).clip(0)

xgb_cv = weighted_log_r2(y_df[TARGET_COLS].values, xgb_oof)
print(f'XGBoost CV: {xgb_cv:.6f}')

In [ ]:
# ── Find optimal blend weight ─────────────────────────────────────────────────

y_arr = y_df[TARGET_COLS].values

def neg_score(w):
    blend = np.clip(w[0], 0, 1) * lgbm_oof + (1 - np.clip(w[0], 0, 1)) * xgb_oof
    return -weighted_log_r2(y_arr, blend)

res = minimize(neg_score, x0=[0.6], bounds=[(0.0, 1.0)], method='L-BFGS-B')
w_lgbm = float(np.clip(res.x[0], 0, 1))
blend_oof = w_lgbm * lgbm_oof + (1 - w_lgbm) * xgb_oof
blend_cv = weighted_log_r2(y_arr, blend_oof)

print(f'\n=== CV RESULTS ===')
print(f'Baseline LightGBM (from reports): 0.716104')
print(f'Tuned LightGBM:  {lgbm_cv:.6f}')
print(f'XGBoost:         {xgb_cv:.6f}')
print(f'Blend ({w_lgbm:.2f}/{1-w_lgbm:.2f}): {blend_cv:.6f}')
print(f'Gain vs baseline: +{blend_cv - 0.716104:.6f}')

print(f'\nPer-target R2 (blend):')
for i, t in enumerate(TARGET_COLS):
    yt = np.log1p(y_arr[:, i].clip(0))
    yp = np.log1p(blend_oof[:, i].clip(0))
    denom = np.sum((yt - yt.mean()) ** 2)
    r2 = 1.0 - np.sum((yt - yp) ** 2) / denom
    print(f'  {t}: {r2:.4f}  (weight={TARGET_WEIGHTS[t]})')

In [ ]:
# ── Train final models on full data ──────────────────────────────────────────

final_lgbm, final_xgb = [], []
for i, target in enumerate(TARGET_COLS):
    y_col = y_df[target].values

    p = {**best_params[target], 'random_state': SEED, 'verbose': -1}
    lgbm = LGBMRegressor(**p)
    lgbm.fit(X, np.log1p(y_col.clip(0)))
    final_lgbm.append(lgbm)

    xgb = XGBRegressor(
        n_estimators=1500, learning_rate=0.02, max_depth=5,
        min_child_weight=3, subsample=0.85, colsample_bytree=0.85,
        reg_alpha=0.1, reg_lambda=1.0, objective='reg:squarederror',
        random_state=SEED, verbosity=0,
    )
    xgb.fit(X, np.log1p(y_col.clip(0)))
    final_xgb.append(xgb)

print('Final models trained on full data')

In [ ]:
# ── Generate test predictions & submission ────────────────────────────────────

test_long = pd.read_csv(DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

test_meta_cols = [c for c in test_long.columns if c not in {'target_name', 'sample_id', 'image_path'}]
test_meta = test_long.groupby('image_path')[test_meta_cols].first().reset_index()
test_feat, _ = build_features(test_meta, fit_stats=stats)

drop_test = set(TARGET_COLS) | {'image_path', 'Sampling_Date', 'sample_id', 'target', 'target_name'}
x_test_df = test_feat[[c for c in test_feat.columns if c not in drop_test]].copy()
x_test_df = pd.get_dummies(x_test_df, columns=[c for c in x_test_df.columns if x_test_df[c].dtype == object], drop_first=False)

# Align columns with training
for col in x_df.columns:
    if col not in x_test_df.columns:
        x_test_df[col] = 0
x_test_df = x_test_df[x_df.columns]
x_test_df = x_test_df.fillna(x_df.median(numeric_only=True))
X_test = x_test_df.values.astype(float)

test_preds = np.zeros((len(test_meta), len(TARGET_COLS)))
for i in range(len(TARGET_COLS)):
    lgbm_p = np.expm1(final_lgbm[i].predict(X_test)).clip(0)
    xgb_p = np.expm1(final_xgb[i].predict(X_test)).clip(0)
    test_preds[:, i] = w_lgbm * lgbm_p + (1 - w_lgbm) * xgb_p

pred_wide = test_meta[['image_path']].copy()
for i, t in enumerate(TARGET_COLS):
    pred_wide[t] = test_preds[:, i]

pred_long = pred_wide.melt(id_vars='image_path', value_vars=TARGET_COLS, var_name='target_name', value_name='target')
sub = test_long[['sample_id', 'image_path', 'target_name']].merge(pred_long, on=['image_path', 'target_name'], how='left')[['sample_id', 'target']]
sub['target'] = sub['target'].clip(lower=0)
sub.to_csv(OUT_DIR / 'submission.csv', index=False)

print('submission.csv saved')
sub

In [ ]:
# ── Save results ──────────────────────────────────────────────────────────────

results = {
    'cv_weighted_r2': blend_cv,
    'lgbm_cv': lgbm_cv,
    'xgb_cv': xgb_cv,
    'lgbm_weight': w_lgbm,
    'baseline_cv': 0.716104,
    'gain_vs_baseline': blend_cv - 0.716104,
    'per_target_r2': {}
}
for i, t in enumerate(TARGET_COLS):
    yt = np.log1p(y_arr[:, i].clip(0))
    yp = np.log1p(blend_oof[:, i].clip(0))
    denom = np.sum((yt - yt.mean()) ** 2)
    results['per_target_r2'][t] = round(1.0 - np.sum((yt - yp) ** 2) / denom, 6)

with open(OUT_DIR / 'cv_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))